In [ ]:
from socrata_interface.domain import Domain
import socrata_interface.transformers as transform
from models.data_summary import DataSummary

In [ ]:
weho = Domain("data.weho.org")
weho_summary = DataSummary()

In [ ]:
ids = weho.dataset_ids()

In [ ]:
for id in ids:
    meta = weho.metadata(id)
    relevant = transform.extract_relevant_metadata(meta)
    
    if relevant.get("display") == "table":
        schema = transform.extract_schema(meta)
        row_counts = weho.row_count(id)
        null_counts = weho.null_count(id, schema)
        sparseness = transform.extract_sparseness(row_counts, null_counts)
        tabular_metadata = transform.aggregate_tabular_metadata(schema, row_counts, sparseness)
        relevant.update(tabular_metadata)

    weho_summary.ingest(relevant)
        

In [ ]:
from socrata_interface.io import write_data

write_data(str(weho), "12345", "summary", weho_summary.to_dict())